In [ ]:
import numpy as np
import pandas as pd

# Semilla para reproducibilidad
np.random.seed(42)
n_registros = 5000

# 1. Generación de variables base
id_deudor = [f"DEU-{i:04d}" for i in range(1, n_registros + 1)]
monto_original = np.round(np.random.lognormal(mean=8.5, sigma=0.8, size=n_registros), 2) # Entre ~$1,000 y ~$30,000 MXN
dias_vencimiento = np.random.randint(1, 180, size=n_registros)
incumplimientos_previos = np.random.poisson(lam=1.2, size=n_registros)

# Pagos parciales: más probables si no tiene tantos incumplimientos
tiene_abono = np.random.binomial(1, p=0.45, size=n_registros)
porcentaje_abonado = np.where(tiene_abono == 1, np.random.uniform(0.1, 0.8, size=n_registros), 0.0)
pagos_parciales = np.round(monto_original * porcentaje_abonado, 2)
monto_pendiente = np.round(monto_original - pagos_parciales, 2)

# Contactabilidad
intentos_contacto = np.random.randint(0, 12, size=n_registros)
respuesta_contacto = []
for intentos in intentos_contacto:
    if intentos == 0:
        respuesta_contacto.append(0)
    else:
        # Probabilidades: [0: sin respuesta, 1: sin compromiso, 2: con compromiso]
        respuesta_contacto.append(np.random.choice([0, 1, 2], p=[0.45, 0.35, 0.20]))
respuesta_contacto = np.array(respuesta_contacto)

# Situación del adeudo: 90% disponible para cobro (1), 10% requiere revisión (0)
situacion_adeudo = np.random.choice([0, 1], size=n_registros, p=[0.10, 0.90])

# 2. Lógica latente para generar el Target ('recuperado': 1 o 0) con sentido financiero
# Calculamos un puntaje oculto de propensión de pago + algo de ruido natural
score_oculto = (
    - 0.015 * dias_vencimiento
    - 0.35 * incumplimientos_previos
    + 1.8 * porcentaje_abonado
    + np.where(respuesta_contacto == 2, 1.5, np.where(respuesta_contacto == 1, 0.3, -0.5))
    - 0.05 * np.maximum(0, intentos_contacto - 5) # Penaliza insistencia excesiva sin éxito
    + np.random.normal(0, 0.6, size=n_registros) # Ruido realista
)

# Convertimos el score a probabilidad usando función sigmoide
probabilidad_pago = 1 / (1 + np.exp(-score_oculto))
recuperado = np.random.binomial(1, p=probabilidad_pago)

# 3. Ensamblar DataFrame
df_entrenamiento = pd.DataFrame({
    'id_deudor': id_deudor,
    'monto_pendiente': monto_pendiente,
    'dias_vencimiento': dias_vencimiento,
    'incumplimientos_anteriores': incumplimientos_previos,
    'pagos_parciales': pagos_parciales,
    'intentos_contacto': intentos_contacto,
    'respuesta_contacto': respuesta_contacto,
    'situacion_adeudo': situacion_adeudo,
    'recuperado': recuperado  # Variable objetivo para entrenar el modelo
})

# Guardar dataset de entrenamiento y un CSV de prueba (sin la columna 'recuperado') para la demo
df_entrenamiento.to_csv('dataset_cobranza_train.csv', index=False)
df_entrenamiento.drop(columns=['recuperado']).sample(30).to_csv('prueba_deudores_demo.csv', index=False)

print("Datasets generados con éxito.")
print(df_entrenamiento.head())
print("\nDistribución de la variable objetivo (recuperado):")
print(df_entrenamiento['recuperado'].value_counts(normalize=True))

In [ ]:
from google.colab import files

files.download('dataset_cobranza_train.csv')
files.download('prueba_deudores_demo.csv')